> 在线讲义：[第 9 章 · 可微 world model：学一个能求导的「物理模拟器」](https://wikieden.github.io/robot-motion-rl/lessons/09-world-model.html) · 本书《机器人运控与强化学习》

# 第 9 章 · 可微 world model：学一个能求导的「物理模拟器」

配套讲义：仓库里 `robot-motion-rl/notes/lessons/09-world-model.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（玩具系统和 d̂ 是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第三步 误差滚大 | $|0.97^k - 0.95^k|$ | `rollout_error(0.95, 0.97, 1, k)` | 0.020、0.120、0.184 |
| 第三步 损失 | $\sum_{j\le k}(\hat x_j - x_j)^2$ | `k_step_loss(..., 1)`、`(..., 8)` | 0.0004、0.0532 |
| 第四步 生存权重 | $c_{t+1} = c_t(1-\hat d)$ | `survival_weights([0, 0.1, 0.5])` | [1, 1, 0.9, 0.45] |

玩具系统（编的）：真实 x_{t+1} = 0.95·x_t，模型学成 0.97·x_t，起点 x_0 = 1。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `rollout`**：线性系统 x_{t+1} = a·x_t 往前滚 steps 步，返回 [x_0, …, x_steps]。

In [ ]:
def rollout(a, x0, steps):
    """线性系统 x_{t+1} = a·x_t 往前滚 steps 步，返回 [x_0, …, x_steps]。"""
    return x0 * a ** np.arange(steps + 1)

**函数 `rollout_error`**：模型自己滚 k 步（吃自己的预测）和真实值的差。

In [ ]:
def rollout_error(a_true, a_model, x0, k):
    """模型自己滚 k 步（吃自己的预测）和真实值的差。"""
    return float(abs(rollout(a_model, x0, k)[-1] - rollout(a_true, x0, k)[-1]))

**函数 `k_step_loss`**：k 步自回归损失：Σ_{j=1..k} (x̂_j − x_j)²，x̂ 从真实起点出发、一路用自己的预测。

In [ ]:
def k_step_loss(a_true, a_model, x0, k):
    """k 步自回归损失：Σ_{j=1..k} (x̂_j − x_j)²，x̂ 从真实起点出发、一路用自己的预测。"""
    xh, x = rollout(a_model, x0, k), rollout(a_true, x0, k)
    return float(np.sum((xh[1:] - x[1:]) ** 2))

**函数 `survival_weights`**：生存权重：c_0 = 1，c_{t+1} = c_t·(1 − d̂_{t+1})。d_hat 给的是 d̂_1, d̂_2, …

In [ ]:
def survival_weights(d_hat):
    """生存权重：c_0 = 1，c_{t+1} = c_t·(1 − d̂_{t+1})。d_hat 给的是 d̂_1, d̂_2, …"""
    c = [1.0]
    for d in d_hat:
        c.append(c[-1] * (1 - d))
    return c

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 误差滚大")
for k in (1, 8, 32):
    print(f"滚 {k} 步：误差 {rollout_error(0.95, 0.97, 1.0, k):.4f}")

In [ ]:
print("\n第二步 1 步损失 vs 8 步损失")
print(f"1 步：{k_step_loss(0.95, 0.97, 1.0, 1):.6f}；8 步：{k_step_loss(0.95, 0.97, 1.0, 8):.6f}")

In [ ]:
print("\n第三步 生存权重")
c = survival_weights([0.0, 0.1, 0.5])
print("d̂ = [0, 0.1, 0.5] → c =", [round(v, 3) for v in c])
print("每步奖励都是 1 时，加权回报 = c_0 + c_1 + c_2 =", round(sum(c[:3]), 3))

## 3. 代码里才看得到的

- 误差滚到 32 步只有 0.184，没有一直变大：因为 0.95 和 0.97 都小于 1，两条轨迹最后都衰减到 0。真实的机器人动力学不会这么「自己收敛」，所以 SuperMimic 要靠 k 步训练和 critic（第 10 章）把视野控制住。
- 生存权重列表比 d̂ 多一个（$c_0 = 1$ 不打折），所以 `[0, 0.1, 0.5]` 对应 4 个 c。

## 4. 自己改一改

- 把真实系数改成 1.01（不稳定的系统），看误差滚 32 步变成多少。
- 用 1 步损失和 8 步损失分别「拟合」：在 0.90 到 1.00 之间扫模型系数，给真实数据加一点噪声，看两种损失选出的系数哪个滚 32 步更准。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 收音机拆不开，听反应画近似图，同一副身体一个模型；
> 知道的随机当输入、在外加，推搡学不了就去掉；
> 闭眼走路误差滚，八步自回归逼它纠偏；
> 摔倒变概率，生存权重连乘，开关变旋钮。